In [1]:
import boto3
import sagemaker
from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.spark.processing import PySparkProcessor
from sagemaker import get_execution_role

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
# Configuración base
REGION = "us-east-1"
ROLE = get_execution_role()
SESSION = sagemaker.Session()

In [ ]:
# Buckets
BRONZE_BUCKET = "hymmrec-dilkehousebronze01"
SILVER_BUCKET = "hymmrec-dilkehousesilver01"
GOLD_BUCKET = "hymmrec-dilkehousegold01"
PLATINUM_BUCKET = "hymmrec-sagemaker-assets"  # Para datos de entrenamiento

# Paths S3 Silver (input)
S3_SILVER_RATINGS = f"s3://{SILVER_BUCKET}/data/obt_movie_affinity/cleansed_ratings/data/"
S3_SILVER_MOVIES = f"s3://{SILVER_BUCKET}/data/obt_movie_affinity/cleansed_movies/data/"
S3_SILVER_POSTERS = f"s3://{SILVER_BUCKET}/data/imv_movie_affinity/movie_posters/"

# Paths S3 Gold (feature store / embeddings)
S3_GOLD_FEATURES = f"s3://{GOLD_BUCKET}/data/ml_feature_store/interactions/"
S3_GOLD_ENCODERS = f"s3://{PLATINUM_BUCKET}/hymmrec/model_artefacts/encoders/"
S3_GOLD_EMBEDDINGS = f"s3://{PLATINUM_BUCKET}/hymmrec/model_artefacts/embeddings/"

# Paths S3 Platinum (training datasets)
S3_PLATINUM_DATASETS = f"s3://{PLATINUM_BUCKET}/hymmrec/datasets/"

# Scripts
PROCESSING_JOB_1_SCRIPT = "processing-feature-eng-job.py"
PROCESSING_JOB_2_SCRIPT = "processing-prepare-data-splits.py"
SCRIPTS_S3_PREFIX = f"s3://{PLATINUM_BUCKET}/sagemaker-scripts/"

print(f"Role: {ROLE}")
print(f"Region: {REGION}")
print(f"Silver: {SILVER_BUCKET}")
print(f"Gold: {GOLD_BUCKET}")
print(f"Platinum: {PLATINUM_BUCKET}")

Role: arn:aws:iam::697682206292:role/sgmkr-notebook-tfm-hymm-rec-ml-iar-dev
Region: us-east-1
Silver: hymmrec-dilkehousesilver01
Gold: hymmrec-dilkehousegold01
Platinum: hymmrec-sagemaker-assets


In [8]:
import os

local_scripts_path = "../dev/"
s3_client = boto3.client("s3")

for script in [PROCESSING_JOB_1_SCRIPT, PROCESSING_JOB_2_SCRIPT]:
    local_path = os.path.join(local_scripts_path, script)
    s3_key = f"sagemaker-scripts/{script}"
    s3_client.upload_file(local_path, PLATINUM_BUCKET, s3_key)
    print(f"Uploaded: {local_path} → s3://{PLATINUM_BUCKET}/{s3_key}")

Uploaded: ../dev/processing-feature-eng-job.py → s3://hymmrec-sagemaker-assets/sagemaker-scripts/processing-feature-eng-job.py
Uploaded: ../dev/processing-prepare-data-splits.py → s3://hymmrec-sagemaker-assets/sagemaker-scripts/processing-prepare-data-splits.py


In [9]:
print("=" * 60)
print("PROCESSING JOB 1: Feature Engineering + Feature Store")
print("=" * 60)

# Usamos PySparkProcessor para la parte de Spark
pyspark_processor = PySparkProcessor(
    role=ROLE,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    framework_version="3.3",
    sagemaker_session=SESSION,
    base_job_name="hymmrec-feature-engineering",
    tags=[
        {"Key": "project", "Value": "hymmrec"},
        {"Key": "phase", "Value": "feature-engineering"},
    ],
)

PROCESSING JOB 1: Feature Engineering + Feature Store


In [ ]:
# Ejecutar Processing Job 1
pyspark_processor.run(
    submit_app=f"{local_scripts_path}/{PROCESSING_JOB_1_SCRIPT}",
    submit_py_files=[],
    arguments=[
        "--region", REGION,
        "--feature-group-name", "hymmrec-interactions-sm-fg",
    ],
    inputs=[
        ProcessingInput(
            source=S3_SILVER_RATINGS,
            destination="/opt/ml/processing/input/ratings",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
        ),
        ProcessingInput(
            source=S3_SILVER_MOVIES,
            destination="/opt/ml/processing/input/movies",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/encoders",
            destination=S3_GOLD_ENCODERS,
            output_name="encoders",
        ),
        ProcessingOutput(
            source="/opt/ml/processing/output/feature_interactions",
            destination=S3_GOLD_FEATURES,
            output_name="feature_interactions",
        ),
    ],
    spark_event_logs_s3_uri=f"s3://{PLATINUM_BUCKET}/spark-logs/job1/",
    logs=True,
    wait=True,
)

print("✅ Processing Job 1 completado.")

INFO:sagemaker:Creating processing-job with name hymmrec-feature-engineering-2026-06-17-01-27-35-116


............


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 # Ejecutar Processing Job 1                                                                 │
│ ❱  2 pyspark_processor.run(                                                                      │
│    3 │   submit_app=f"{local_scripts_path}/{PROCESSING_JOB_1_SCRIPT}",                           │
│    4 │   submit_py_files=[],                                                                     │
│    5 │   arguments=[                                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/spark/processing.py │
│ :922 in run                                                                                      │
│                                                                                                  │
│    919 │   │   │   spark_event_logs_s3_uri=spark_event_logs_s3_uri,                              │
│    920 │   │   )                                                                                 │
│    921 │   │                                                                                     │
│ ❱  922 │   │   return super().run(                                                               │
│    923 │   │   │   submit_app=submit_app,                                                        │
│    924 │   │   │   inputs=extended_inputs,                                                       │
│    925 │   │   │   outputs=extended_outputs,                                                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                   

In [14]:

# Ejecutar Processing Job 1
pyspark_processor.run(
    submit_app=f"{local_scripts_path}/{PROCESSING_JOB_1_SCRIPT}",
    submit_py_files=[],
    arguments=[
        "--region", REGION,
        "--feature-group-name", "hymmrec-interactions-sm-fg",
    ],
    inputs=[
        ProcessingInput(
            source=S3_SILVER_RATINGS,
            destination="/opt/ml/processing/input/ratings",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
        ),
        ProcessingInput(
            source=S3_SILVER_MOVIES,
            destination="/opt/ml/processing/input/movies",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
        ),
        ProcessingInput(
            source=S3_SILVER_POSTERS,
            destination="/opt/ml/processing/input/posters",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/embeddings",
            destination=S3_GOLD_EMBEDDINGS,
            output_name="embeddings",
        ),
        ProcessingOutput(
            source="/opt/ml/processing/output/encoders",
            destination=S3_GOLD_ENCODERS,
            output_name="encoders",
        ),
        ProcessingOutput(
            source="/opt/ml/processing/output/feature_interactions",
            destination=S3_GOLD_FEATURES,
            output_name="feature_interactions",
        ),
    ],
    spark_event_logs_s3_uri=f"s3://{PLATINUM_BUCKET}/spark-logs/job1/",
    logs=True,
    wait=True,
)

print("✅ Processing Job 1 completado.")

INFO:sagemaker:Creating processing-job with name hymmrec-feature-engineering-2026-06-17-01-33-40-463


..

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 # Ejecutar Processing Job 1                                                                 │
│ ❱  2 pyspark_processor.run(                                                                      │
│    3 │   submit_app=f"{local_scripts_path}/{PROCESSING_JOB_1_SCRIPT}",                           │
│    4 │   submit_py_files=[],                                                                     │
│    5 │   arguments=[                                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/spark/processing.py │
│ :922 in run                                                                                      │
│                                                                                                  │
│    919 │   │   │   spark_event_logs_s3_uri=spark_event_logs_s3_uri,                              │
│    920 │   │   )                                                                                 │
│    921 │   │                                                                                     │
│ ❱  922 │   │   return super().run(                                                               │
│    923 │   │   │   submit_app=submit_app,                                                        │
│    924 │   │   │   inputs=extended_inputs,                                                       │
│    925 │   │   │   outputs=extended_outputs,                                                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                   